# Vanishing and Exploding Gradients

During backpropagation, gradients travel from the loss toward the earlier layers.

- **Vanishing:** gradients become tiny → early layers barely learn
- **Exploding:** gradients become huge → parameter updates become unstable

![Gradient propagation through network layers](diagrams/09-gradient-propagation.svg)

Each step multiplies the gradient by another local derivative. Repeated small multipliers shrink it; repeated large multipliers grow it.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

# A tiny example of gradients being multiplied through five steps
for multiplier in [0.2, 2.0]:
    x = torch.tensor(1.0, requires_grad=True)
    value = x

    for _ in range(5):
        value = value * multiplier

    value.backward()  # stores d(value)/d(x) in x.grad
    print(f"{multiplier} multiplied 5 times -> gradient: {x.grad.item():.5f}")

```text
0.2 × 0.2 × 0.2 × 0.2 × 0.2 = 0.00032  <- vanishing
2.0 × 2.0 × 2.0 × 2.0 × 2.0 = 32       <- exploding
```

A real network uses matrices and many paths, but the multiplication idea is the same.

## Three Ways to Improve Gradient Stability

![Three ways to improve gradient stability](diagrams/09-gradient-stability.svg)

## 1. Weight Initialization

Every `nn.Linear` starts with random weights. Their scale matters because each layer calculates:

```text
output = input @ weight.T + bias
```

Weights that are badly scaled can repeatedly shrink or enlarge values and gradients.

In [ ]:
torch.manual_seed(42)
layer = nn.Linear(9, 16)

print(f"Weight shape: {layer.weight.shape}")  # 16 neurons, 9 weights each
print(f"Minimum: {layer.weight.min().item():.3f}")
print(f"Maximum: {layer.weight.max().item():.3f}")

### He / Kaiming Initialization

Kaiming initialization is designed for layers followed by `ReLU` or similar activations. The trailing underscore means the existing tensor is changed in place.

In [ ]:
relu_layer = nn.Linear(9, 16)

nn.init.kaiming_uniform_(
    relu_layer.weight,
    nonlinearity="relu",  # match the activation used after this layer
)
nn.init.zeros_(relu_layer.bias)

print(f"Initialized weight standard deviation: {relu_layer.weight.std().item():.3f}")

Use the initialization that matches the **activation after the layer**:

| Activation after the layer | Common initialization |
|---|---|
| ReLU / Leaky ReLU | Kaiming (He) |
| Sigmoid / Tanh | Xavier (Glorot) |

Kaiming is normally used for the ReLU hidden layers—not automatically for every layer.

### Initialization Inside a Custom Model

The layers are created first, then their starting weights are replaced.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        # fc1 and fc2 are followed by ReLU
        nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity="relu")
        nn.init.kaiming_uniform_(self.fc2.weight, nonlinearity="relu")

        # fc3 is followed by sigmoid, so Xavier is a better match
        nn.init.xavier_uniform_(self.fc3.weight)

        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))


model = Net()
sample = torch.randn(4, 9)
print(f"Input shape:  {sample.shape}")
print(f"Output shape: {model(sample).shape}")

> When training with `BCEWithLogitsLoss`, remove the final sigmoid from `forward()`. The loss applies it internally; use `torch.sigmoid(model(x))` later when you want probabilities.

## 2. Activation Functions

ReLU helps positive values keep a gradient of `1`, but negative values get a gradient of `0`. A neuron that stays on the negative side can stop learning—the **dying ReLU** problem.

ELU keeps a non-zero slope for negative inputs, so those neurons can still receive updates.

In [ ]:
values = torch.tensor([-2.0, -0.5, 0.5, 2.0])

for activation_name, activation in [("ReLU", F.relu), ("ELU", F.elu)]:
    x = values.clone().requires_grad_(True)
    output = activation(x)
    output.sum().backward()  # get one gradient for every input value

    print(f"{activation_name} output:   {output.detach()}")
    print(f"{activation_name} gradient: {x.grad}\n")

## 3. Batch Normalization

Batch normalization works on each feature across the current batch:

```text
1. center:     value - batch mean
2. normalize:  centered value / batch standard deviation
3. adjust:     normalized value * learned scale + learned shift
```

The learned scale and shift mean the model is not forced to keep every feature at exactly mean `0` and standard deviation `1`.

The usual order is:

![Linear, batch-normalization, and activation order](diagrams/09-batch-normalization-order.svg)

`BatchNorm1d(16)` uses `16` because the preceding linear layer returns 16 values per sample. None of these operations changes that feature count.

In [ ]:
class StableNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(9, 16)
        self.bn1 = nn.BatchNorm1d(16)  # must match fc1's 16 outputs
        self.fc2 = nn.Linear(16, 8)
        self.bn2 = nn.BatchNorm1d(8)   # must match fc2's 8 outputs
        self.fc3 = nn.Linear(8, 1)

        nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity="relu")
        nn.init.kaiming_uniform_(self.fc2.weight, nonlinearity="relu")
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = self.fc1(x)  # [batch, 9] -> [batch, 16]
        x = self.bn1(x)  # normalize the 16 features across the batch
        x = F.relu(x)

        x = self.fc2(x)  # [batch, 16] -> [batch, 8]
        x = self.bn2(x)  # normalize the 8 features across the batch
        x = F.relu(x)

        return self.fc3(x)  # raw logit for BCEWithLogitsLoss


stable_model = StableNet()
batch = torch.randn(5, 9)
logits = stable_model(batch)
print(f"Batch shape:  {batch.shape}")
print(f"Logits shape: {logits.shape}")

### Why `model.train()` and `model.eval()` Matter

Batch normalization behaves differently in the two modes:

- `model.train()` uses the current batch statistics and updates its running statistics.
- `model.eval()` uses the saved running statistics for stable predictions.

Like Dropout, BatchNorm makes switching modes important.

## Quick Look

| Problem or tool | Meaning | Main idea |
|---|---|---|
| Vanishing gradients | Gradients approach zero | Early layers barely update |
| Exploding gradients | Gradients become huge | Updates become unstable |
| Kaiming initialization | Starting weights for ReLU-like activations | Keep value and gradient scales reasonable |
| ELU | Activation with a negative-side slope | Helps avoid dying neurons |
| BatchNorm | Normalizes each feature across a batch, then learns scale and shift | Stabilizes values during training |

```python
x = self.fc1(x)
x = self.bn1(x)
x = F.relu(x)
```